# FD002 and FD004 teachers on the original baseline pipeline

Train an additive-attention BiGRU, a GRU with skip connections, and a residual CNN on exactly the original prepared 30-cycle archives. The runner verifies SHA-256 fingerprints. All models use the original cap-125 training labels, MSE objective, and capped validation RMSE for checkpoint selection. Select a single model or a fixed equal-weight ensemble on validation. Then evaluate the frozen choices on the same official test engine windows as the baselines. Compare raw test RMSE in cycles, with capped RMSE displayed separately.

In [ ]:
from pathlib import Path
import subprocess
import sys
import torch
from google.colab import drive

if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime for this experiment.")
drive.mount("/content/drive")
project = Path("/content/drive/MyDrive/ML projec/SpikeDistill-RUL")
repo = Path("/content/ML3103-matched-teachers")
if not (repo / ".git").exists():
    subprocess.run(["git", "clone", "https://github.com/abhav-arora/ML3103-project.git", str(repo)], check=True)
subprocess.run(["git", "rev-parse", "HEAD"], cwd=repo, check=True)

data_root = project / "artifacts/data"
output = project / "colab_runs/matched_baseline_teachers"
subsets = ["FD002", "FD004"]
command = [sys.executable, "-u", "-m", "research_extensions.v2.matched_teachers",
           "--data-root", str(data_root), "--output", str(output), "--epochs", "18"]

In [ ]:
for phase in ("train", "freeze"):
    for subset in subsets:
        subprocess.run([*command, "--subset", subset, "--phase", phase], cwd=repo, check=True)

In [ ]:
for subset in subsets:
    subprocess.run([*command, "--subset", subset, "--phase", "test"], cwd=repo, check=True)

In [ ]:
import pandas as pd

scores = pd.concat([pd.read_csv(output / subset / "test_scores.csv").assign(subset=subset)
                    for subset in subsets], ignore_index=True)
baseline = pd.read_csv(repo / "results/test_metrics_per_seed.csv")
baseline = baseline.query("seed == 42 and target == 'raw' and subset in @subsets")
baseline = baseline.loc[baseline.groupby("subset").RMSE.idxmin(), ["subset", "model", "RMSE"]]
baseline = baseline.rename(columns={"model": "best_original_baseline", "RMSE": "baseline_raw_rmse"})
comparison = scores.merge(baseline, on="subset", validate="many_to_one")
comparison["delta_vs_baseline"] = comparison["raw_rmse"] - comparison["baseline_raw_rmse"]
display(comparison.round(3))
comparison.to_csv(output / "comparison_with_original_baselines.csv", index=False)